In [31]:
import os
import time
import pandas as pd
from ytmusicapi import YTMusic

PLAYLIST_LIMIT=500
PLAYLIST_SONG_LIMIT=10000
yt = YTMusic('../headers_auth.json')

def parse_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])  # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_playlist(yt, playlist_meta, print_meta=False):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    if print_meta: 
        print(pd.DataFrame.from_dict(playlist_meta, orient='index'))
    tracks = parse_tracks(track_list)
    return tracks, playlist_meta

def create_rating_playlist_subset(tracks, name, rating):
    assert rating in ('LIKE', 'DISLIKE', 'INDIFFERENT')
    filtered_tracks = tracks.loc[tracks['likeStatus'] == rating]
    video_ids = filtered_tracks['videoId'].unique().tolist()
    if len(video_ids) > 0:
        pl_id = yt.create_playlist(
            title=name + ' ' + rating.lower(), 
            description='generated from %s includes %s subset' % (name, rating),
            privacy_status='PRIVATE', 
            video_ids=video_ids
        )
        print('Created %s playlist with id %s' % (rating, pl_id))

In [32]:
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
print('Playlists:\n %s' % sorted(playlists.sort_values('title')['title']))


Playlists:
 ['Ambient Psychill', 'Ambient Unrated Albums 2018-2019', 'Beat instrumentals', 'Beats Without Rhymes like', 'Beats indie Chill like', 'Beats indie Chill radio', 'Blues delta roots radio like', 'Bossa Nova like', 'Bossa Nova radio', 'Brass n chill', 'Chillwave', 'Doo wop pop  youtube radio', 'Electronic 2010s like', 'Electronic Focus like', 'Electronic Focus radio', 'Electronic House Special like', 'Electronic House Special radio', 'Electronic Innerwaves like', 'Electronic Innerwaves radio', "Electronic We're Alone Now like", "Electronic We're Alone Now radio", 'Folk like', 'Folk radio', 'Grunge like', 'Grunge radio', 'Hip Hop 1990s like', 'Hip Hop 1990s radio', 'Hip Hop 2000s like', 'Hip Hop 2000s radio', 'Hip Hop Classic West Coast like', 'Hip Hop Classic West Coast radio', 'Hip Hop Hits liked', 'Hip Hop Hits unrated', 'Hip hop It Was a Good Day like', 'Hip hop It Was a Good Day radio', 'Hiphop southeast Ride Around Shining', 'Indie 1990s Rock like', 'Indie 1990s Rock radi

In [34]:
for i, row in playlists.iterrows():
    if 'indifferent' in row.title:
        print(row.title)

In [19]:
# # Example: Create Unrated and Liked Subset Playlist

# playlist_name = 'Analog Grooves'
# playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
# metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
# tracks, metadata = parse_playlist(yt, metadata)
# print('Selected Playlist:\n%s' % metadata)

# create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
# create_rating_playlist_subset(tracks, playlist_name, 'LIKE')

In [4]:
# Example: Group public playlists
public_playlists = {}
privacy = 'PUBLIC'
for i, p in playlists.iterrows():
    if i == 0: continue # skip giant likes playlist
    if 'z_' in p or 'zz_' in p or 'zzz_' in p:
        continue
    metadata = yt.get_playlist(p['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    if metadata['privacy'] == privacy:
        public_playlists[p['playlistId']] = p['title']
        print('Found %s playlist named: %s' % (privacy.lower(), p['title']))

list(public_playlists.values())

Found public playlist named: '60s Folk Rock
Found public playlist named: Classic Rock Instrumentals
Found public playlist named: Hauntingly Beautiful
Found public playlist named: Instrumental Soul Grooves
Found public playlist named: Soul-Soaked Samples
Found public playlist named: Soulful Organ Instrumentals
Found public playlist named: Stiff Drink in a Dim Place
Found public playlist named: Your Songs: Singer-Songwriter Classics


["'60s Folk Rock",
 'Classic Rock Instrumentals',
 'Hauntingly Beautiful',
 'Instrumental Soul Grooves',
 'Soul-Soaked Samples',
 'Soulful Organ Instrumentals',
 'Stiff Drink in a Dim Place',
 'Your Songs: Singer-Songwriter Classics']

In [23]:
# For each playlist, Create Unrated and Liked Subset Playlist, delete original
playlist_names = [
    'x_r.indie_tracks_radio', 'x_r.hiphop_tracks_radio', 'x_r.futuregarage_tracks_radio', 'x_r.futurefunkairlines_tracks_radio', 'x_r.futurebeats_tracks_radio', 'x_r.futurebass_tracks_radio', 'x_r.chillwave_tracks_radio', 'x_r.chillmusic_tracks_radio', 'x_r.blues_tracks_radio', 'x_r.90shiphop_tracks_radio', 
    'trip hop radio', 'soul radio', 'soul 1960s radio', 'rock classic radio', 'rock 2000s radio', 'rock 1990s built to spill radio', 'rock 1970s classic radio', 'rock 1960s classic radio', 'Reggae 1970 roots radio', 'punk 1970s radio', 'psychedelic classic rock radio', 
    'psych rock radio', 'Post-Punk 1970s-1980s radio', 'Oldies radio', 'nu disco radio', 'Jukebox Vintage Party radio', 'jazz cool radio', 'Indie radio', 'indie loose live chill radio', 'Indie 2000s radio', 'Indie 1990s Rock radio', 
    'hiphop old school radio', 'hiphop modern radio', 'Hip Hop Hits unrated', 'Hip Hop 2000s radio', 'Hip Hop 1990s radio', 'garage rock radio', 'future beats radio', 'folk 1960s radio', 'electronic radio', 'Electronic House Special radio', 'blues radio', 
    'beats radio', 'soul motown radio', 'pop singer songwriters radio', 'electronic Dance radio', 'Beats indie Chill radio', 'Electronic Innerwaves radio', 'Folk radio', 'Indie Folk radio', 'Jazz Noir radio', 'Shoegaze radio', 'doo wop radio', 'rock 1950s roots radio'
]
#  'x_r.treemusic radio', 'x_r.reggae_tracks_radio', 'x_r.realdubstep_tracks_radio', 'x_r.psychedelicrock_tracks_radio', 'x_r.nudisco_tracks_radio', 'x_r.jazzyhiphop_tracks_radio', 'x_r.jazznoir_tracks_radio', 'x_r.jazz_tracks_radio', 'x_r.indierock_tracks_radio', 
for playlist_name in playlist_names:
    print('Sorting %s in to like and indifferent' % playlist_name)
    playlist = playlists.loc[playlists['title'] == playlist_name].iloc[0] # first match
    metadata = yt.get_playlist(playlist['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    tracks, metadata = parse_playlist(yt, metadata)
    create_rating_playlist_subset(tracks, playlist_name, 'INDIFFERENT')
    time.sleep(20)
    create_rating_playlist_subset(tracks, playlist_name, 'LIKE')
    time.sleep(20)

Sorting x_r.jazzyhiphop_tracks_radio in to like and indifferent
Created INDIFFERENT playlist with id PLWptjpDqazOy_DFSH79lUMUKG3gPZ-Sqy
Created LIKE playlist with id PLWptjpDqazOzODeIjDDwZ25ybhYstLSvs
Sorting x_r.jazznoir_tracks_radio in to like and indifferent
Created INDIFFERENT playlist with id PLWptjpDqazOyc9Ht46kf7YoVtPULT_fv2
Sorting x_r.jazz_tracks_radio in to like and indifferent
Created INDIFFERENT playlist with id PLWptjpDqazOxwUg9kkIIDnbkVunaDb5rP
Created LIKE playlist with id PLWptjpDqazOwP2h7vkKIh_Xva7H5dZI_B
Sorting x_r.indierock_tracks_radio in to like and indifferent
Created INDIFFERENT playlist with id PLWptjpDqazOw6iNCNhyHMisKDMsDgRqF1
Created LIKE playlist with id PLWptjpDqazOzIAdHpZuKlj_sOTAiVrqpw
Sorting x_r.indie_tracks_radio in to like and indifferent
Created INDIFFERENT playlist with id PLWptjpDqazOwmzYhM_dHGvDgB3Dpeyz8h
Created LIKE playlist with id PLWptjpDqazOx6ZjkF-4Oh2yRS1qeRMwAQ
Sorting x_r.hiphop_tracks_radio in to like and indifferent
Created INDIFFERENT

In [ ]:
# RECENT
#  'beats Soulful Instrumentals & Pensive like', 'folk 1960s like', 
# 'rock classic instrumentals radio', 'Electronic House Special radio', 'Folk radio' , 
# 'ambient Dream Pop Deep Sleep like', 'ambient haunting harmonious', 'Oldies radio',
# 'Reggae 1970 roots radio', 'Reggae sunshine wailer relaxation radio', 'blues radio', 
# 'rock classic instrumentals radio', 'Bossa Nova', 'jazz cool radio', 'pop singer songwriterss', 
# 'doo wop indifferent', 'soul motown radio', 'soul 1960s radio', 'ambient haunting harmonious radio',
# 'soul radio', 'jukebox Vintage Rock Instrumentals radio',
# 'hiphop modern radio', 'hiphop old school radio', 'Bossa Nova radio'
# 'Hip hop 1990s NY', 'Psychedelic Indie radio', 'jazz cool radio', 'Indie radio', 
# 'Rock progressive', 'indie loose live chill radio','Jukebox Vintage Party radio',
#  'Unexpected Best Night Ever', 'punk 1970s British radio', 'Classic Rock Summer', 
# 'nu disco radio', 'beats radio', 'hiphop old school', 'punk 1970s radio', 
# 'Easy-Listening Acid Trip', 'Reggae Dub', 'Post-Punk 1970s-1980s radio',
#  'Frequencies', 'Monterey Pop Festival 1967', 'Take It Slow', 'Shoegazing', 

# additional_playlist_names = [
#     'Beats indie Chill indifferent',
#     "Folk radio",
#     "Electronic Focus radio",
#     "Electronic House Special radio",
#     "Electronic We're Alone Now radio",
#     "electronic radio",
#     "future beats radio",
#     "blues radio",
#     'hiphop old radio',
#     'indie loose live chill radio',
#     "Indie Hazy Summer indifferent",
#     'Hip hop It Was a Good Day indifferent',
#     'nu disco radio',
#     'Indie radio',
#     'jazz cool radio',
#     'jazz gloom smooth',
#     'jazz solo guitar radio',
#     'psychedelic classic rock radio',
#     'indie subreddit',
#     'futurebeats subbreddit',
#     'hiphop subreddit',
#     'r.treemusic',
#     'rock classic radio',
#     'rock 1960s classic radio',
#     'rock 1970s classic radio',
#     'Rock 1967-1969 radio',
#     'soul radio',
#     'Soul Classic Sunshine radio'
# ]

In [30]:
# # restore a tsv playlist
# videoIds = list(pd.read_csv('..\\playlists\\jukebox Vintage Rock Instrumentals like.tsv', sep='\t', index_col=0).videoId.unique())
# pl_id = yt.create_playlist(
#     title='x_r.chillmusic_like', 
#     description='re-generated from x_r.chillmusic_like.tsv',
#     privacy_status='PRIVATE', 
#     video_ids=videoIds
# )